In [ ]:
# run_analysis_v12_microscore.py

import os
import json
import re
from datetime import datetime
from typing import List, Dict, Optional

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# --- Step 1: Define the new, nested Pydantic models ---
class ScoreComponent(BaseModel):
    """A single component of a score calculation for a piece of evidence."""
    reason: str = Field(description="The reason for assigning these points (e.g., 'mentions specific financial impact').")
    points: int = Field(description="The points for this specific reason (can be positive or negative).")

class ScoreCalculation(BaseModel):
    """The detailed breakdown of how a snippet's score was calculated."""
    components: List[ScoreComponent] = Field(description="A list of all scoring components.")
    final_snippet_score: int = Field(description="The final calculated score for this snippet, which is the sum of all component points.")

class Evidence(BaseModel):
    """A single piece of evidence with a detailed micro-scorecard."""
    snippet: str = Field(description="The exact, original sentence or phrase from the document.")
    score_calculation: ScoreCalculation = Field(description="The detailed micro-scorecard for this snippet.")
    reasoning_summary: str = Field(description="A brief summary explaining the final score.")

class EvidenceBasedReport(BaseModel):
    """The final structured output from the LLM."""
    holistic_analysis: str = Field(description="The comprehensive 'Chain of Thought' analysis paragraph.")
    evidence_list: List[Evidence] = Field(description="A list of all evidence snippets, each with its own micro-scorecard.")

# --- Helper functions (loading configs, parsing filename) ---
def load_risk_events(filepath: str = "./Data/LLM/risk_events_9.json") -> List[Dict]:
    """从JSON文件加载风险事件。"""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"错误：加载风险事件文件 '{filepath}' 失败: {e}")
        return []

def load_prompt_template(filepath: str = "./Data/LLM/prompt_template_v5.txt") -> str:
    """从文本文件加载Prompt模板。"""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError as e:
        print(f"错误：加载Prompt模板文件 '{filepath}' 失败: {e}")
        return ""

def parse_filename(filepath: str) -> Dict[str, Optional[str]]:
    """解析像 'AAPL_2022-10-28.txt' 这样的文件名以提取公司和日期。"""
    try:
        basename = os.path.basename(filepath)
        match = re.match(r"([A-Z]+)_(\d{4}-\d{2}-\d{2})\.txt", basename)
        if match:
            company_name = match.group(1)
            filing_date = match.group(2)
            datetime.strptime(filing_date, "%Y-%m-%d")
            return {"company_name": company_name, "filing_date": filing_date}
    except (ValueError, IndexError):
        pass
    print(f"警告：无法从文件名 '{filepath}' 中解析公司和日期。将使用默认值。")
    return {"company_name": "Unknown", "filing_date": "Unknown"}

# --- Main analysis function ---
def run_full_analysis(document_filepath: str):
    load_dotenv()
    """主执行函数，运行完整的微观记分卡分析流程。"""
    if not os.getenv("OPENAI_API_KEY"):
        print("错误：OPENAI_API_KEY 未设置。请在运行脚本前将其设置为环境变量。")
        return

    custom_risk_events = load_risk_events()
    prompt_template_string = load_prompt_template()
    file_info = parse_filename(document_filepath)
    
    if not custom_risk_events or not prompt_template_string:
        print("由于配置或模板文件缺失，无法继续。")
        return

    llm = ChatOpenAI(model="o3", temperature=1, model_kwargs={"response_format": {"type": "json_object"}})
    parser = JsonOutputParser(pydantic_object=EvidenceBasedReport)
    
    prompt = PromptTemplate(
        template=prompt_template_string,
        input_variables=["document_text", "company_name", "filing_date", "event_name", "event_timeframe", "event_description"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    
    chain = prompt | llm | parser

    try:
        with open(document_filepath, "r", encoding="utf-8") as f:
            document_content = f.read()
    except FileNotFoundError:
        print(f"错误：在 '{document_filepath}' 未找到文档文件。")
        return

    all_final_reports = []
    print(f"--- 开始为文档 '{document_filepath}' 进行分析 ---")
    print(f"--- 公司: {file_info['company_name']}, 财报日期: {file_info['filing_date']} ---")
    
    for i, event in enumerate(custom_risk_events, 1):
        print(f"\n({i}/{len(custom_risk_events)}) 正在分析事件: {event['event_name']}...")
        try:
            evidence_report = chain.invoke({
                "document_text": document_content,
                "company_name": file_info["company_name"],
                "filing_date": file_info["filing_date"],
                "event_name": event["event_name"],
                "event_timeframe": event["event_timeframe"],
                "event_description": event["event_description"]
            })
            
            evidence_list = evidence_report.get('evidence_list', [])
            num_evidence = len(evidence_list)

            # Python端现在从每条证据的 'final_snippet_score' 累加总分
            total_score = sum(e['score_calculation']['final_snippet_score'] for e in evidence_report['evidence_list'])
            final_score = max(0, min(100, total_score))

             # 计算平均严重性，如果证据数量不为0
            average_severity = (total_score / num_evidence) if num_evidence > 0 else 0

            all_final_reports.append({
                "event_info": event,
                "holistic_analysis": evidence_report['holistic_analysis'],
                "evidence_list": evidence_report['evidence_list'],
                "calculated_score": final_score,
                "average_severity": average_severity
            })
            print(f"  > 分析完成. 找到 {num_evidence} 条证据. 总分: {final_score}, 平均严重性: {average_severity:.2f}")

        except Exception as e:
            print(f"  > 在分析此事件时发生错误: {e}")
    
    print_summary_report(all_final_reports, file_info)

# --- Updated report printing function ---
def print_summary_report(reports: List[Dict], file_info: Dict):
    """用于打印带有微观记分卡的详细摘要报告的辅助函数。"""
    if not reports:
        print("\n无分析结果可显示。")
        return

    print("\n\n======================================================")
    print(f"  为 {file_info['company_name']} ({file_info['filing_date']}) 生成的微观记分卡报告")
    print("======================================================")
    
    sorted_reports = sorted(reports, key=lambda x: x.get('calculated_score', 0), reverse=True)

    for report in sorted_reports:
        event = report['event_info']
        score = report['calculated_score']
        avg_sev = report['average_severity']
        holistic_analysis = report['holistic_analysis']
        evidence_list = report['evidence_list']

        print(f"\n--- 事件: {event.get('event_name', 'N/A')} ---")
        print(f"  总冲击力得分 (0-100): {score}")
        print(f"  平均证据强度 (0-10): {avg_sev:.2f}")
        print(f"\n  整体分析 (思维链):")
        print(f"    {holistic_analysis}")
        
        if evidence_list:
            sorted_evidence = sorted(evidence_list, key=lambda x: x.get('score_calculation', {}).get('final_snippet_score', 0), reverse=True)
            print("\n  证据分解 (按严重性排序):")
            for ev in sorted_evidence:
                calc = ev.get('score_calculation', {})
                snippet_score = calc.get('final_snippet_score', 0)
                print(f"    [总分: {snippet_score}/10] \"{ev.get('snippet', '')}\"")
                print(f"      摘要: {ev.get('reasoning_summary', '')}")
                print(f"      记分卡:")
                for comp in calc.get('components', []):
                    sign = "+" if comp.get('points', 0) >= 0 else ""
                    print(f"        [{sign}{comp.get('points')}] {comp.get('reason')}")
        else:
            print("\n  证据分解: 未找到相关证据。")
            
    print("\n======================================================")

if __name__ == "__main__":
    run_full_analysis(document_filepath='AAPL_2023-11-03.txt')

--- 开始为文档 'AAPL_2023-11-03.txt' 进行分析 ---
--- 公司: AAPL, 财报日期: 2023-11-03 ---

(1/9) 正在分析事件: September_11_Attacks...
  > 分析完成. 找到 4 条证据. 总分: 6, 平均严重性: 1.50

(2/9) 正在分析事件: Global_Financial_Crisis_Subprime...
  > 分析完成. 找到 5 条证据. 总分: 12, 平均严重性: 2.40

(3/9) 正在分析事件: Arab_Spring...
  > 分析完成. 找到 3 条证据. 总分: 5, 平均严重性: 1.67

(4/9) 正在分析事件: Paris_Agreement_on_Climate_Change...
  > 分析完成. 找到 6 条证据. 总分: 25, 平均严重性: 4.17

(5/9) 正在分析事件: Brexit_Referendum...
  > 分析完成. 找到 7 条证据. 总分: 22, 平均严重性: 3.14

(6/9) 正在分析事件: COVID_19_Pandemic...
  > 分析完成. 找到 3 条证据. 总分: 14, 平均严重性: 4.67

(7/9) 正在分析事件: Russia_Ukraine_Conflict...
  > 分析完成. 找到 5 条证据. 总分: 10, 平均严重性: 2.00

(8/9) 正在分析事件: US_China_Chip_Export_Controls...
  > 分析完成. 找到 10 条证据. 总分: 61, 平均严重性: 6.10

(9/9) 正在分析事件: Israel_Hamas_Conflict...
  > 分析完成. 找到 5 条证据. 总分: 9, 平均严重性: 1.80


  为 AAPL (2023-11-03) 生成的微观记分卡报告

--- 事件: US_China_Chip_Export_Controls ---
  总冲击力得分 (0-100): 61
  平均证据强度 (0-10): 6.10

  整体分析 (思维链):
    The filing was prepared roughly one year after the U